# 🧪 W10-D2 Trace 不是日志：从线性事件到因果树

> 配套阅读：同名 `.md`。本 notebook 只用小规模、可重复的模拟来验证核心治理约束。

**实验目标：** 构造 Span 树，比较只按时间排序的日志与可查询的父子因果关系。


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

from dataclasses import dataclass
from collections import defaultdict

@dataclass
class Span:
    span_id: str; parent_id: str | None; name: str; started: int; ended: int; status: str = "ok"
    @property
    def duration(self): return self.ended - self.started

spans = [Span("s0", None, "channel_dispatch", 0, 1250), Span("s1", "s0", "rag_retrieval", 20, 340),
         Span("s2", "s1", "rag_rerank", 120, 200), Span("s3", "s0", "llm", 360, 1210),
         Span("s4", "s0", "send_reply", 1215, 1240)]
for s in sorted(spans, key=lambda x: x.started):
    print(f"LOG {s.started:4d}ms {s.name:18s} completed")
print("\n日志按时间可读，但没有父子关系；以下单元用 parent_id 还原结构。")


In [ ]:
children = defaultdict(list)
by_id = {s.span_id: s for s in spans}
for s in spans: children[s.parent_id].append(s)

def show_tree(parent=None, depth=0):
    for child in children[parent]:
        print("  " * depth + f"├─ {child.name} ({child.duration}ms, {child.status})")
        show_tree(child.span_id, depth + 1)

show_tree()
slowest = max(spans, key=lambda s: s.duration)
print(f"\n可查询结论：最慢 span = {slowest.name}, {slowest.duration}ms")


In [ ]:
names = [s.name for s in spans]
durations = [s.duration for s in spans]
colors = ["#E15759" if s.name == "llm" else "#4C78A8" for s in spans]
plt.figure(figsize=(8, 3.5))
plt.barh(names, durations, color=colors)
plt.xlabel("耗时 (ms)"); plt.title("同一 trace 的 Span 耗时：LLM 是主要瓶颈")
plt.tight_layout(); plt.show()

# Audit 是扁平治理证据，和 trace 的过程结构职责不同。
audit = {"actor": "tenant_admin", "action": "skill_release:invoke", "result": "succeeded", "trace_id": "demo-001"}
print("Audit 记录：", audit)
